<a href="https://colab.research.google.com/github/nchandu-how/ai-ml-genai-npl/blob/main/Medical_Analyst.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [1]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1
!pip install llama-cpp-python

# Installation for CPU llama-cpp-python
# uncomment and run the following code in case GPU is not being used
# !CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python==0.1.85 --force-reinstall --no-cache-dir -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 MB 14.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 6.2 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.16-cp312-cp312-linux_x86_64.whl size=4503286 sha256=df0bc8cfbddd772c6821f591ce8d184c6ea59698705ee42a06e325b0966eae13
  Stored in directory: /root/.cache/pip/wheels/90/82/ab/8784ee3fb99ddb07fd36a679ddbe63122cc07718f6c1eb3be8
Successfully built llama-cpp-python


In [2]:
!pip install numpy
!pip install pandas

In [3]:
# For installing the libraries & downloading models from HF Hub
%pip install huggingface_hub
%pip install tiktoken
%pip install pymupdf
%pip install langchain
%pip install langchain-community
%pip install chromadb
%pip install sentence-transformers



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 83.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 78.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.7/502.7 kB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 7.2 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.16
    Uninstalling langchain-core-1.2.16:
      Successfully uninstalled langchain-core-1.2.16
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
googl

In [4]:
#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

## Question Answering using LLM

#### Downloading and Loading the model

In [5]:
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_basename = "mistral-7b-instruct-v0.2.Q6_K.gguf"


model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
)

print(f"Model downloaded to: {model_path}")

llm = Llama(
    model_path=model_path,
    n_ctx=2300,
    n_gpu_layers=8,
    n_batch=128
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


mistral-7b-instruct-v0.2.Q6_K.gguf:   0%|          | 0.00/5.94G [00:00<?, ?B/s]

llama_model_loader: loaded meta data with 24 key-value pairs and 291 tensors from /root/.cache/huggingface/hub/models--TheBloke--Mistral-7B-Instruct-v0.2-GGUF/snapshots/3a6fbf4a41a1d52e415a4958cde6856d34b2db93/mistral-7b-instruct-v0.2.Q6_K.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = mistralai_mistral-7b-instruct-v0.2
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loade

Model downloaded to: /root/.cache/huggingface/hub/models--TheBloke--Mistral-7B-Instruct-v0.2-GGUF/snapshots/3a6fbf4a41a1d52e415a4958cde6856d34b2db93/mistral-7b-instruct-v0.2.Q6_K.gguf


load: control token:      2 '</s>' is not marked as EOG
load: control token:      1 '<s>' is not marked as EOG
load: special_eos_id is not in special_eog_ids - the tokenizer config may be incorrect
load: printing all EOG tokens:
load:   - 2 ('</s>')
load: special tokens cache size = 3
load: token to piece cache size = 0.1637 MB
print_info: arch             = llama
print_info: vocab_only       = 0
print_info: n_ctx_train      = 32768
print_info: n_embd           = 4096
print_info: n_layer          = 32
print_info: n_head           = 32
print_info: n_head_kv        = 8
print_info: n_rot            = 128
print_info: n_swa            = 0
print_info: is_swa_any       = 0
print_info: n_embd_head_k    = 128
print_info: n_embd_head_v    = 128
print_info: n_gqa            = 4
print_info: n_embd_k_gqa     = 1024
print_info: n_embd_v_gqa     = 1024
print_info: f_norm_eps       = 0.0e+00
print_info: f_norm_rms_eps   = 1.0e-05
print_info: f_clamp_kqv      = 0.0e+00
print_info: f_max_alibi_bias = 0.

#### Response

In [6]:
# Temparature is kept at 0 because for this business use case we want factual answers

def response(query,max_tokens=200,temperature=0,top_p=0.95,top_k=50):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [7]:
query1 = "What is the protocol for managing sepsis in a critical care unit?"
print(response(query1))


llama_perf_context_print:        load time =    6334.37 ms
llama_perf_context_print: prompt eval time =    6330.92 ms /    16 tokens (  395.68 ms per token,     2.53 tokens per second)
llama_perf_context_print:        eval time =  158201.84 ms /   199 runs   (  794.98 ms per token,     1.26 tokens per second)
llama_perf_context_print:       total time =  164662.40 ms /   215 tokens
llama_perf_context_print:    graphs reused =        192




Sepsis is a life-threatening condition that can arise from an infection, and it requires prompt recognition and aggressive management in a critical care unit. The following are the general steps for managing sepsis in a critical care unit:

1. Early recognition and suspicion: Septic patients may present with non-specific symptoms such as fever, chills, tachycardia, tachypnea, altered mental status, or lactic acidosis. It is essential to have a high index of suspicion for sepsis, especially in patients with known infections or risk factors.
2. Initial assessment and resuscitation: The first step in managing sepsis is to assess and resuscitate the patient. This includes assessing airway, breathing, circulation, and disability (ABCD) and providing appropriate interventions such as oxygen therapy, fluid resuscitation, and vasopressor support if necessary.
3. Source control:


### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [8]:
query2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
print(response(query2))

Llama.generate: 2 prefix-match hit, remaining 32 prompt tokens to eval
llama_perf_context_print:        load time =    6334.37 ms
llama_perf_context_print: prompt eval time =   13344.45 ms /    32 tokens (  417.01 ms per token,     2.40 tokens per second)
llama_perf_context_print:        eval time =  157127.24 ms /   199 runs   (  789.58 ms per token,     1.27 tokens per second)
llama_perf_context_print:       total time =  170598.32 ms /   231 tokens
llama_perf_context_print:    graphs reused =        192




Appendicitis is a medical condition characterized by inflammation of the appendix, a small pouch that extends from the cecum, the first part of the large intestine. The symptoms of appendicitis can vary from person to person, but the following are the most common:

1. Abdominal pain: The pain is typically located in the lower right side of the abdomen and may be dull at first, but it can quickly become sharp and severe. The pain may worsen when you move, cough, or sneeze.
2. Loss of appetite: You may lose your appetite due to the abdominal pain or feel nauseous.
3. Nausea and vomiting: You may feel sick to your stomach and vomit.
4. Fever: A fever of 100.4°F (38°C) or higher is common with appendicitis.
5. Const


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [9]:
query3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
print(response(query3))

Llama.generate: 4 prefix-match hit, remaining 34 prompt tokens to eval
llama_perf_context_print:        load time =    6334.37 ms
llama_perf_context_print: prompt eval time =   14465.42 ms /    34 tokens (  425.45 ms per token,     2.35 tokens per second)
llama_perf_context_print:        eval time =  161064.44 ms /   199 runs   (  809.37 ms per token,     1.24 tokens per second)
llama_perf_context_print:       total time =  175660.01 ms /   233 tokens
llama_perf_context_print:    graphs reused =        192




Sudden patchy hair loss, also known as alopecia areata, is a common autoimmune disorder that affects the hair follicles, leading to hair loss in small, round patches on the scalp, beard, or other areas of the body. The exact cause of alopecia areata is not known, but it is believed to be related to a problem with the immune system.

There are several treatments that have been shown to be effective in addressing sudden patchy hair loss:

1. Corticosteroids: Corticosteroids are anti-inflammatory medications that can help reduce inflammation and suppress the immune system, allowing the hair follicles to regrow. They can be applied topically or taken orally.
2. Minoxidil: Minoxidil is a medication that has been shown to promote hair growth in some people with alopecia areata. It is applied


### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [10]:
query4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
print(response(query4))

Llama.generate: 2 prefix-match hit, remaining 28 prompt tokens to eval
llama_perf_context_print:        load time =    6334.37 ms
llama_perf_context_print: prompt eval time =   12009.02 ms /    28 tokens (  428.89 ms per token,     2.33 tokens per second)
llama_perf_context_print:        eval time =  159371.53 ms /   199 runs   (  800.86 ms per token,     1.25 tokens per second)
llama_perf_context_print:       total time =  171509.17 ms /   227 tokens
llama_perf_context_print:    graphs reused =        191




A person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function, is typically diagnosed with a traumatic brain injury (TBI). The treatment for a TBI depends on the severity and location of the injury, as well as the individual's overall health and age.

Immediate treatment for a TBI may include:

1. Emergency medical care: This may include surgery to remove hematomas or other obstructions, as well as treatment for other injuries.
2. Medications: Depending on the symptoms, medications may be prescribed to manage pain, reduce swelling, prevent or treat infections, or control seizures.
3. Rehabilitation: Rehabilitation may include physical therapy, occupational therapy, speech therapy, and cognitive rehabilitation to help the person regain lost skills and functions.
4. Supportive care: This may include assistance with daily activities


### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [11]:
query5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
print(response(query5))

Llama.generate: 2 prefix-match hit, remaining 35 prompt tokens to eval
llama_perf_context_print:        load time =    6334.37 ms
llama_perf_context_print: prompt eval time =   14660.99 ms /    35 tokens (  418.89 ms per token,     2.39 tokens per second)
llama_perf_context_print:        eval time =  159450.69 ms /   199 runs   (  801.26 ms per token,     1.25 tokens per second)
llama_perf_context_print:       total time =  174239.30 ms /   234 tokens
llama_perf_context_print:    graphs reused =        192




First and foremost, if a person has fractured their leg during a hiking trip, it is essential to ensure their safety and prevent further injury. Here are some necessary precautions and treatment steps:

1. Assess the situation: Check the extent of the injury and assess the person's condition. If the fracture is open or the person is in severe pain, immobilize the leg with a splint or a makeshift sling to prevent any movement.
2. Call for help: If possible, call for emergency medical assistance. If there is no cell phone reception, try to signal for help using a mirror, whistle, or other means.
3. Provide first aid: Apply a sterile dressing to the injury to prevent infection. If the fracture is open, apply pressure to stop any bleeding.
4. Immobilize the leg: Use a splint, a makeshift sling, or a tour


## Question Answering using LLM with Prompt Engineering

In [12]:
# Add instructions to the prompt for better response generation
# Given the sensitivity of medical diagnosis, I am adding a prompt to minimize hallucination
system_prompt = "You are a helpful and knowledgeable medical assistant. Answer the following medical question accurately and concisely based on common medical knowledge. If you don't know the answer, please state that you don't have enough information."
print ("System Prompt:" + system_prompt)

System Prompt:You are a helpful and knowledgeable medical assistant. Answer the following medical question accurately and concisely based on common medical knowledge. If you don't know the answer, please state that you don't have enough information.


### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [13]:
query1 = system_prompt + "\n" + "What is the protocol for managing sepsis in a critical care unit?"
print(response(query1))


Llama.generate: 1 prefix-match hit, remaining 61 prompt tokens to eval
llama_perf_context_print:        load time =    6334.37 ms
llama_perf_context_print: prompt eval time =   26008.43 ms /    61 tokens (  426.37 ms per token,     2.35 tokens per second)
llama_perf_context_print:        eval time =  159463.14 ms /   199 runs   (  801.32 ms per token,     1.25 tokens per second)
llama_perf_context_print:       total time =  185599.55 ms /   260 tokens
llama_perf_context_print:    graphs reused =        191




Sepsis is a life-threatening condition caused by a severe infection. In a critical care unit, managing sepsis involves the following steps:

1. Early recognition and diagnosis: Monitor vital signs, laboratory values, and clinical symptoms closely. Suspect sepsis in any patient with suspected or confirmed infection and organ dysfunction.
2. Immediate fluid resuscitation: Administer intravenous fluids to maintain adequate blood pressure and organ perfusion. The goal is to achieve a mean arterial pressure (MAP) of 65 mmHg or higher.
3. Antibiotics: Administer broad-spectrum antibiotics as soon as possible based on the suspected infection site and microbiological culture results.
4. Source control: Identify and address the source of infection, such as removing an infected catheter or draining an abscess.
5. Vasopressors: If fluid res


### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [14]:
query2 = system_prompt + "\n" + "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
print(response(query2))

Llama.generate: 48 prefix-match hit, remaining 32 prompt tokens to eval
llama_perf_context_print:        load time =    6334.37 ms
llama_perf_context_print: prompt eval time =   13802.72 ms /    32 tokens (  431.34 ms per token,     2.32 tokens per second)
llama_perf_context_print:        eval time =  160115.36 ms /   199 runs   (  804.60 ms per token,     1.24 tokens per second)
llama_perf_context_print:       total time =  174045.00 ms /   231 tokens
llama_perf_context_print:    graphs reused =        192




Appendicitis is a common inflammatory condition of the appendix, a small pouch located in the lower right side of the abdomen. The symptoms of appendicitis can include:

1. Sudden pain in the lower right abdomen, which may start as a mild ache and gradually develop into a sharp pain.
2. Loss of appetite and feeling sick to your stomach (nausea).
3. Fever, which may be low-grade at first but can rise as high as 103°F (39.4°C).
4. Vomiting.
5. Constipation or diarrhea.
6. Abdominal swelling and tenderness.
7. Inability to pass gas or have a bowel movement.
8. Pain in the lower back, on the right side.
9. Feeling restless or unable to find a comfortable position.


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [15]:
query3 = system_prompt + "\n" + "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
print(response(query3))

Llama.generate: 50 prefix-match hit, remaining 34 prompt tokens to eval
llama_perf_context_print:        load time =    6334.37 ms
llama_perf_context_print: prompt eval time =   14409.92 ms /    34 tokens (  423.82 ms per token,     2.36 tokens per second)
llama_perf_context_print:        eval time =  160843.15 ms /   199 runs   (  808.26 ms per token,     1.24 tokens per second)
llama_perf_context_print:       total time =  175382.91 ms /   233 tokens
llama_perf_context_print:    graphs reused =        192




Sudden patchy hair loss, also known as alopecia areata, is an autoimmune condition that causes hair loss in small, round patches on the scalp, beard, or other areas of the body. The exact cause is unknown, but it's believed to be related to a problem with the immune system.

Effective treatments for addressing sudden patchy hair loss include:

1. Corticosteroids: These are anti-inflammatory medications that can help reduce inflammation and suppress the immune system response. They can be applied topically or taken orally.
2. Immunotherapy: This involves the use of medications that stimulate the immune system to attack the hair loss. One such medication is minoxidil.
3. Hair transplantation: This is a surgical procedure in which healthy hair is transplanted from one area of the scalp to another. It's usually considered a


### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
query4 = system_prompt + "\n" + "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
print(response(query4))

Llama.generate: 48 prefix-match hit, remaining 28 prompt tokens to eval


### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
query5 = system_prompt + "\n" + "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
print(response(query5))

## Data Preparation for RAG

### Loading the Data

In [ ]:
## Data Preparation for RAG
### Loading the Data
#Libraries for processing dataframes, text
import json, os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

### Data Overview

In [20]:
## Data Overview
from google.colab import drive
drive.mount('/content/drive')

pdf_path = "/content/drive/MyDrive/Colab Notebooks/medical_diagnosis_manual.pdf"

if not os.path.exists(pdf_path):
    raise FileNotFoundError(f"PDF not found at {pdf_path}")

# Use the PyMuPDFLoader to load the document
try:
    # Note: Loading large PDFs (>4,000 pages) may require significant memory. Consider chunked processing if RAM is limited.
    loader = PyMuPDFLoader(pdf_path)
    documents = loader.load()
    print(f"Loaded {len(documents)} pages from the PDF.")
except Exception as e:
    print(f"Error loading PDF: {e}")
    raise

Mounted at /content/drive
Loaded 4114 pages from the PDF.


#### Checking the first 5 pages

In [21]:
# Preview first 5 pages (or fewer if PDF is smaller) for debugging
for i in range(min(5, len(documents))):
    print(f"--- Page {i+1} ---")
    print(documents[i].page_content[:500] + "...")

#### Checking the number of pages
print(f"Number of pages: {len(documents)}")

--- Page 1 ---
naveen.k.chandu@gmail.com
WNMXIBC4UH
This file is meant for personal use by naveen.k.chandu@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action....
--- Page 2 ---
naveen.k.chandu@gmail.com
WNMXIBC4UH
This file is meant for personal use by naveen.k.chandu@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action....
--- Page 3 ---
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    ....................................
--- Page 4 ---
491
Chapter 44. Foot & Ankle Disorders    ..............

#### Checking the number of pages

### Data Chunking

In [22]:
## Function to do data chunking

def get_data_chunks(
    data,
    chunk_size=1000,
    chunk_overlap=200,
    split_method="recursive",
    separators=["\n\n", "\n", ". ", " ", ""],
    min_chunk_size=50,
    respect_sentence_boundaries=True,
    respect_paragraph_boundaries=True,
    length_function=len,
    max_chunks=None,
    add_metadata=True
):
    """
    Splits a list of documents into smaller chunks using a specified method.

    Args:
        data: A list of document objects (e.g., from Langchain loaders).
        chunk_size: The maximum size of each chunk.
        chunk_overlap: The number of characters to overlap between chunks.
        split_method: The method to use for splitting ('recursive').
        separators: A list of separators to use for splitting.
        min_chunk_size: The minimum size of each chunk.
        respect_sentence_boundaries: Whether to try to split on sentence boundaries.
        respect_paragraph_boundaries: Whether to try to split on paragraph boundaries.
        length_function: The function to use to measure chunk length.
        max_chunks: The maximum number of chunks to generate.
        add_metadata: Whether to add metadata to the chunks.

    Returns:
        A list of chunked documents.
    """
    try:
        if split_method == "recursive":
            text_splitter = RecursiveCharacterTextSplitter(
                chunk_size=chunk_size,
                chunk_overlap=chunk_overlap,
                length_function=length_function,
                is_separator_regex=False,
                separators=separators
            )
            chunks = text_splitter.split_documents(data)
        else:
            raise ValueError(f"Unsupported split_method: {split_method}")
    except Exception as e:
        print(f"Error during chunking: {e}")
        raise
    chunks = [chunk for chunk in chunks if length_function(chunk.page_content) >= min_chunk_size]
    if add_metadata:
        for chunk in chunks:
            if not chunk.metadata:
                chunk.metadata = {"source": "medical_diagnosis_manual.pdf", "page": 0}  # Fallback
    if max_chunks is not None:
        chunks = chunks[:max_chunks]
    return chunks

In [23]:

# Utilize the get_data_chunks function for the loaded PDF
chunk_size = 500  # chunk size
chunk_overlap = 100 # chunk overlap

chunks = get_data_chunks(
    documents,
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    split_method="recursive",
    separators=["\n\n", "\n", ". ", " ", ""],
    min_chunk_size=50,
    respect_sentence_boundaries=True,
    respect_paragraph_boundaries=True,
    length_function=len,
    max_chunks=None, # Process all chunks
    add_metadata=True
)

# Print some validation and check statements
print(f"\n--- Chunking Validation ---")
print(f"Number of documents loaded: {len(documents)}")
print(f"Number of chunks created: {len(chunks)}")

# Check the first few chunks
print(f"\nFirst {min(5, len(chunks))} chunks:")
for i in range(min(5, len(chunks))):
  print(f"--- Chunk {i+1} ---")
  print(f"Chunk length: {len(chunks[i].page_content)}")
  print(f"Chunk metadata: {chunks[i].metadata}")
  print(chunks[i].page_content[:200] + "...") # Print first 200 characters of the chunk content

# Check the last few chunks (if there are more than 5)
if len(chunks) > 5:
    print(f"\nLast {min(5, len(chunks)-5)} chunks:")
    for i in range(max(0, len(chunks)-5), len(chunks)):
        print(f"--- Chunk {i+1} ---")
        print(f"Chunk length: {len(chunks[i].page_content)}")
        print(f"Chunk metadata: {chunks[i].metadata}")
        print(chunks[i].page_content[:200] + "...") # Print first 200 characters of the chunk content

# Additional checks
if len(chunks) > 0:
    # Check minimum chunk size
    min_len = min(len(chunk.page_content) for chunk in chunks)
    print(f"\nMinimum chunk length: {min_len}")
    if min_len < 50: # Based on min_chunk_size parameter
        print("Warning: Some chunks might be smaller than the specified minimum size.")

    # Check for empty chunks
    empty_chunks = sum(1 for chunk in chunks if len(chunk.page_content) == 0)
    print(f"Number of empty chunks: {empty_chunks}")

    # Check metadata presence (assuming add_metadata is True)
    metadata_missing = sum(1 for chunk in chunks if not hasattr(chunk, 'metadata') or not chunk.metadata)
    print(f"Number of chunks missing metadata: {metadata_missing}")
else:
    print("\nNo chunks were created.")


--- Chunking Validation ---
Number of documents loaded: 4114
Number of chunks created: 34563

First 5 chunks:
--- Chunk 1 ---
Chunk length: 186
Chunk metadata: {'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creator': 'Atop CHM to PDF Converter', 'creationdate': '2012-06-15T05:44:40+00:00', 'source': '/content/drive/MyDrive/Colab Notebooks/medical_diagnosis_manual.pdf', 'file_path': '/content/drive/MyDrive/Colab Notebooks/medical_diagnosis_manual.pdf', 'total_pages': 4114, 'format': 'PDF 1.7', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-03-04T02:29:10+00:00', 'trapped': '', 'modDate': 'D:20260304022910Z', 'creationDate': 'D:20120615054440Z', 'page': 0}
naveen.k.chandu@gmail.com
WNMXIBC4UH
This file is meant for personal use by naveen.k.chandu@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action....
--- Chunk 2 ---
Chunk length: 186
Chunk metadata: 

### Embedding

In [24]:
# Note: For M4 with 16GB RAM, process chunks in small batches to avoid memory issues.
embedding_function = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

# Embed chunks in batches
embedded_chunks = []
batch_size = 100  # Safe for 16GB RAM
for i in range(0, len(chunks), batch_size):
    batch = chunks[i:i+batch_size]
    for j, chunk in enumerate(batch):
        try:
            embedding = embedding_function.embed_query(chunk.page_content)
            embedded_chunks.append((chunk, embedding))
        except Exception as e:
            print(f"Error embedding chunk {i+j}: {e}")
print(f"\n--- Embedding Validation ---")
print(f"Number of original chunks: {len(chunks)}")
print(f"Number of successfully embedded chunks: {len(embedded_chunks)}")
if len(embedded_chunks) == len(chunks):
    print("All chunks successfully embedded.")
else:
    print(f"Embedded {len(embedded_chunks)}/{len(chunks)} chunks.")

if len(embedded_chunks) > 0:
    first_embedded_chunk, first_embedding = embedded_chunks[0]
    print(f"Type of first embedding: {type(first_embedding)}")
    import numpy as np
    first_embedding_np = np.array(first_embedding)
    print(f"Dimension of first embedding: {len(first_embedding_np)}")
    print(f"First 10 values: {first_embedding_np[:10]}")

/tmp/ipykernel_1972/3405431599.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_function = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


--- Embedding Validation ---
Number of original chunks: 34563
Number of successfully embedded chunks: 34563
All chunks successfully embedded.
Type of first embedding: <class 'list'>
Dimension of first embedding: 384
First 10 values: [-0.04696565  0.01271422  0.00275378 -0.05078879  0.04886311 -0.02082288
  0.03921168  0.03130541 -0.02651285  0.0105603 ]


### Vector Database

In [25]:
#Vector Database
import os, shutil

from langchain_community.vectorstores import Chroma

persist_directory = 'medical_db'
if os.path.exists(persist_directory):
    print(f"Removing existing database at {persist_directory}")
    shutil.rmtree(persist_directory, ignore_errors=True)

if not os.path.exists(persist_directory):
    os.makedirs(persist_directory)
    print(f"Created directory at {persist_directory}")

try:
    vector_db = Chroma.from_documents(
        documents=[chunk for chunk, embedding in embedded_chunks],
        embedding=embedding_function,
        persist_directory=persist_directory
    )
except Exception as e:
    print(f"Error creating Chroma database: {e}")
    raise

# Simplified vector database validation
print(f"\n--- Vector Database Validation ---")
if os.path.exists(persist_directory):
    print(f"Chroma database created at {persist_directory}.")
try:
    count = vector_db._collection.count()
    print(f"Number of items in database: {count}")
    print("Match with embedded chunks." if count == len(embedded_chunks) else "Item count mismatch.")
except Exception as e:
    print(f"Error retrieving database count: {e}")

Created directory at medical_db

--- Vector Database Validation ---
Chroma database created at medical_db.
Number of items in database: 34563
Match with embedded chunks.


### Retriever

In [26]:
# prompt: code a retriever using the above code with the appropriate search method and k value
retriever = vector_db.as_retriever(
    search_type="similarity",  # Using similarity search
    search_kwargs={"k": 3}     # Retrieve top 3 similar documents
)

# --- Validation and Conclusions ---
print(f"\n--- Retriever Validation ---")
if retriever:
    print("Conclusion: Retriever successfully created.")
    try:
        # Test with queries from problem statement (e.g., sepsis, appendicitis, hair loss)
        sample_query = "What are the symptoms of appendicitis?"
        retrieved_docs = retriever.invoke(sample_query)
        print(f"\nRetrieved {len(retrieved_docs)} documents for a sample query.")
        if len(retrieved_docs) > 0:
            print("Sample of retrieved document content (first 100 chars):")
            print(retrieved_docs[0].page_content[:100] + "...")
        else:
            print("No documents retrieved. Check vector database content or query relevance.")

        print(f"\nRetriever configuration:")
        print(f"Search type: {retriever.search_type}")
        print(f"Search kwargs: {retriever.search_kwargs}")
        if retriever.search_kwargs.get("k") == 3:
            print("Conclusion: Retriever configured with correct k value (3).")
        else:
            print(f"Warning: Retriever's k value is {retriever.search_kwargs.get('k')}, expected 3.")
    except ValueError as ve:
        print(f"Database error during retrieval: {ve}")
    except Exception as e:
        print(f"Error during sample query with retriever: {e}")
        print("Conclusion: Retriever might not be configured correctly or database has issues.")
else:
    print("Conclusion: Failed to create the retriever.")


--- Retriever Validation ---
Conclusion: Retriever successfully created.

Retrieved 3 documents for a sample query.
Sample of retrieved document content (first 100 chars):
Symptoms and Signs
The classic symptoms of acute appendicitis are epigastric or periumbilical pain f...

Retriever configuration:
Search type: similarity
Search kwargs: {'k': 3}
Conclusion: Retriever configured with correct k value (3).


### System and User Prompt Template

In [27]:
## 1. The system message describing the assistant's role.
## 2. A user message template including context and the question.

In [28]:
# --- System and User Prompts ---
qna_system_message = "You are a knowledgeable medical assistant. Provide accurate, concise answers based solely on the provided context from the Merck Manuals. If the context is insufficient, state that you lack information."
qna_user_message_template = """Context: {context}\n\nQuestion: {question}\nAnswer concisely and factually."""


In [29]:
# --- Evaluation Prompts ---
groundedness_rater_system_message = "You are an evaluator assessing the groundedness of a medical response. Rate the response based on whether all factual claims are supported by the provided context, on a scale of 1–5 (1: contradicts context, 5: fully supported). Return only the numeric rating."
relevance_rater_system_message = "You are an evaluator assessing the relevance of a medical response. Rate the response based on how directly it addresses the question, on a scale of 1–5 (1: irrelevant, 5: fully relevant). Return only the numeric rating."
user_message_template = """Question: {question}\nResponse: {answer}\nContext: {context}\nRating:"""


In [30]:
# --- Queries to Test ---
queries_to_test = {
    "Query 1": "What is the protocol for managing sepsis in a critical care unit?",
    "Query 2": "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?",
    "Query 3": "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?",
    "Query 4": "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?",
    "Query 5": "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
}


In [31]:
# --- RAG Response Function ---
def generate_rag_response(user_input, retriever, max_tokens=128, temperature=0, top_p=0.95, top_k=50):
    global qna_system_message, qna_user_message_template
    try:
        relevant_document_chunks = retriever.invoke(user_input)
        context_list = [d.page_content for d in relevant_document_chunks]
        context_for_query = ". ".join(context_list)[:4000]  # Limit for M4
        user_message = qna_user_message_template.replace('{context}', context_for_query).replace('{question}', user_input)
        prompt = f"{qna_system_message}\n{user_message}".strip()
        response = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k
        )
        return response['choices'][0]['text'].strip(), context_for_query
    except ValueError as ve:
        return f"Retrieval error: {ve}", ""
    except Exception as e:
        return f"Sorry, I encountered the following error: {e}", ""

In [32]:
# --- Groundedness and Relevance Evaluation Function ---
def generate_ground_relevance_response(user_input, retriever, max_tokens=10, temperature=0, top_p=0.95, top_k=50):
    global groundedness_rater_system_message, relevance_rater_system_message, user_message_template
    try:
        # Generate RAG response and get context
        answer, context_for_query = generate_rag_response(user_input, retriever, max_tokens=128, temperature=0)

        # Groundedness evaluation
        groundedness_prompt = user_message_template.format(
            question=user_input,
            answer=answer,
            context=context_for_query
        )
        groundedness_response = llm(
            prompt=f"{groundedness_rater_system_message}\n{groundedness_prompt}",
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k
        )
        groundedness_rating = groundedness_response['choices'][0]['text'].strip()

        # Relevance evaluation
        relevance_prompt = user_message_template.format(
            question=user_input,
            answer=answer,
            context=context_for_query
        )
        relevance_response = llm(
            prompt=f"{relevance_rater_system_message}\n{relevance_prompt}",
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k
        )
        relevance_rating = relevance_response['choices'][0]['text'].strip()

        return groundedness_rating, relevance_rating, answer, context_for_query
    except Exception as e:
        return f"Error: {e}", f"Error: {e}", "", ""


In [33]:
# --- Fine-tuning Parameters ---
param_combinations = [
    {"chunk_size": 500, "chunk_overlap": 100, "retriever_k": 3, "llm_max_tokens": 128, "llm_temperature": 0},
    {"chunk_size": 750, "chunk_overlap": 150, "retriever_k": 5, "llm_max_tokens": 200, "llm_temperature": 0.1},
    {"chunk_size": 1000, "chunk_overlap": 200, "retriever_k": 3, "llm_max_tokens": 128, "llm_temperature": 0},
    {"chunk_size": 500, "chunk_overlap": 100, "retriever_k": 5, "llm_max_tokens": 200, "llm_temperature": 0.2},
    {"chunk_size": 750, "chunk_overlap": 150, "retriever_k": 3, "llm_max_tokens": 128, "llm_temperature": 0.1}
]


In [ ]:
results = {}
evaluation_results = {}

print("\n--- Fine-tuning RAG Parameters ---")
for i, params in enumerate(param_combinations):
    print(f"\n--- Combination {i+1}: {params} ---")

    # Chunking (assumes get_data_chunks is defined as in previous code)
    current_chunks = get_data_chunks(
        documents,
        chunk_size=params["chunk_size"],
        chunk_overlap=params["chunk_overlap"],
        split_method="recursive",
        separators=["\n\n", "\n", ". ", " ", ""],
        min_chunk_size=50,
        add_metadata=True
    )
    print(f"  Created {len(current_chunks)} chunks.")

    # Embedding
    current_embedded_chunks = []
    batch_size = 50  # Safe for M4
    for j in range(0, len(current_chunks), batch_size):
        chunk_batch = current_chunks[j:j+batch_size]
        try:
            embeddings = embedding_function.embed_documents([chunk.page_content for chunk in chunk_batch])
            for chunk, embedding in zip(chunk_batch, embeddings):
                current_embedded_chunks.append((chunk, embedding))
        except Exception as e:
            print(f"  Error embedding batch {j}: {e}")
    print(f"  Embedded {len(current_embedded_chunks)} chunks.")

    # Vector Database
    persist_directory = f'medical_db_combo_{i+1}'
    if os.path.exists(persist_directory):
        shutil.rmtree(persist_directory, ignore_errors=True)
    os.makedirs(persist_directory, exist_ok=True)

    if current_embedded_chunks:
        try:
            current_vector_db = Chroma.from_documents(
                documents=[chunk for chunk, embedding in current_embedded_chunks],
                embedding=embedding_function,
                persist_directory=persist_directory
            )
            current_retriever = current_vector_db.as_retriever(
                search_type="similarity",
                search_kwargs={"k": params["retriever_k"]}
            )
            print(f"  Retriever created with k={params['retriever_k']}")

            # Query Execution and Evaluation
            results[f"Combination {i+1}"] = {}
            evaluation_results[f"Combination {i+1}"] = {}
            for query_name, query_text in queries_to_test.items():
                print(f"    Testing Query: {query_name}")
                groundedness_rating, relevance_rating, response_text, context = generate_ground_relevance_response(
                    query_text,
                    current_retriever
                )
                results[f"Combination {i+1}"][query_name] = response_text
                evaluation_results[f"Combination {i+1}"][query_name] = {
                    "groundedness_rating": groundedness_rating,
                    "relevance_rating": relevance_rating,
                    "context": context[:200] + "..." if context else "No context"
                }
                print(f"    Response: {response_text[:200]}...")
                print(f"    Groundedness Rating: {groundedness_rating}")
                print(f"    Relevance Rating: {relevance_rating}")
        except Exception as e:
            print(f"  Error creating vector DB or retriever: {e}")
    else:
        print("  No embedded chunks for vector DB.")



--- Fine-tuning RAG Parameters ---

--- Combination 1: {'chunk_size': 500, 'chunk_overlap': 100, 'retriever_k': 3, 'llm_max_tokens': 128, 'llm_temperature': 0} ---
  Created 34563 chunks.
  Embedded 34563 chunks.
  Retriever created with k=3
    Testing Query: Query 1


Llama.generate: 4 prefix-match hit, remaining 400 prompt tokens to eval


In [ ]:
# --- Compare Results ---
print("\n\n--- Comparison of Results ---")
for query_name, query_text in queries_to_test.items():
    print(f"\n### {query_name}: {query_text}")
    for combo_name, query_results in results.items():
        response_text = query_results.get(query_name, "Response not found")
        eval_data = evaluation_results.get(combo_name, {}).get(query_name, {})
        print(f"\n#### {combo_name} (Chunk Size: {param_combinations[int(combo_name.split(' ')[1])-1]['chunk_size']}, "
              f"Overlap: {param_combinations[int(combo_name.split(' ')[1])-1]['chunk_overlap']}, "
              f"Retriever k: {param_combinations[int(combo_name.split(' ')[1])-1]['retriever_k']}, "
              f"LLM Tokens: {param_combinations[int(combo_name.split(' ')[1])-1]['llm_max_tokens']}, "
              f"LLM Temp: {param_combinations[int(combo_name.split(' ')[1])-1]['llm_temperature']})")
        print(f"Response: {response_text}")
        print(f"Groundedness Rating: {eval_data.get('groundedness_rating', 'N/A')}")
        print(f"Relevance Rating: {eval_data.get('relevance_rating', 'N/A')}")
        print(f"Context Preview: {eval_data.get('context', 'N/A')}")
        print("-" * 50)


In [ ]:
# --- Evaluation Summary ---
print("\n--- Evaluation Results Summary ---")
eval_summary = {}
for combo_name in evaluation_results:
    eval_summary[combo_name] = {}
    for query_name in queries_to_test:
        eval_data = evaluation_results[combo_name].get(query_name, {})
        eval_summary[combo_name][query_name] = {
            "Groundedness": eval_data.get("groundedness_rating", "N/A"),
            "Relevance": eval_data.get("relevance_rating", "N/A")
        }
eval_df = pd.DataFrame.from_dict({(c, q): eval_summary[c][q] for c in eval_summary for q in eval_summary[c]}, orient='index')
print(eval_df)